# Reproduce: confidence-guided spectral search for AutoND (Colab)

This notebook clones the project, pins AutoND, runs the test suite and the experiments.
Every experiment writes CSV/JSON plus a `manifest.json` (packages, AutoND commit, hardware, seeds).

* **CPU-only phases** (1, 3a, 4, 5a, E2, E3, E5) need NumPy/SciPy only.
* **Neural phases**: E1 uses the PyTorch DBitNet port, which runs on a Colab GPU if one is attached
  (Runtime -> Change runtime type -> GPU). The original TensorFlow pipeline (phase 2/3b/5b) needs
  TF 2.15 / Python <= 3.11 -- see `scripts/setup_gpu_linux.sh`.

Set `REPO_URL` to wherever you host this project (it is not published by default).

In [ ]:
REPO_URL = "<your-git-url-for-this-project>"   # e.g. a private GitHub repo
!git clone $REPO_URL crypto || true
%cd crypto
!bash scripts/setup_autond.sh
!pip -q install "numpy<2" scipy pandas matplotlib pytest h5py

In [ ]:
import torch, platform; print(platform.python_version(), torch.__version__, torch.cuda.is_available())
!python -m pytest -q

## CPU phases (seeded; minutes to ~1 h each, see README runtime table)

In [ ]:
!python experiments/phase1_verify.py --config experiments/configs/simon32_demo.json
!python experiments/phase3_score_variability.py --config experiments/configs/simon32_demo.json
!python experiments/phase4_validate_scorer.py --config experiments/configs/simon32_demo.json
!python experiments/phase5_scoring_compare.py --config experiments/configs/simon32_demo.json

In [ ]:
!python experiments/novel_e2_detection_depth.py --t 100000 --reps 3
!python experiments/novel_e3_search.py --seeds 1 2 3 --generations 10
!python experiments/novel_e1_score_candidates.py --cipher speck3264
!python experiments/novel_e5_compare.py

## E1: neural validation (GPU recommended; resumable, skips finished candidates)

In [ ]:
!python experiments/novel_e1_neural.py --cipher speck3264 train
!python experiments/novel_e1_neural.py --cipher speck3264 analyse

## Figures (generated only from the CSV/JSON above)

In [ ]:
!python experiments/make_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')): print(f); display(Image(f))